In [1]:
# TASK 3 - DATA CLEANING - Titanic Dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Dataset Load
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print("--- BEFORE CLEANING ---")
print("Shape:", df.shape)
print("\nNull Values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nData Types:\n", df.dtypes)
print("\nSex column ke alag-alag values:")
print(df['Sex'].value_counts(dropna=False))

# 2. Messy Data Create karna (Task me bola hai Male/male/M jaisa issue dikhana hai)
# Real data me ye issue kam hota hai, isliye hum artificially dikhate hai
df.loc[0:5, 'Sex'] = ['male', 'Male', 'M', 'Female', 'F', 'female']

print("\n--- Messy Sex Values ---")
print(df['Sex'].value_counts(dropna=False))

# 3. Cleaning Start
# a) Duplicate Hatana
before_rows = len(df)
df = df.drop_duplicates()
print(f"\nDuplicates hataye: {before_rows - len(df)}")

# b) Sex ko Standardize Karna - Male/male/M -> Male, Female/female/F -> Female
df['Sex'] = df['Sex'].replace({'male':'Male', 'M':'Male', 'm':'Male',
                               'female':'Female', 'F':'Female', 'f':'Female'})
print("\nCleaned Sex Values:")
print(df['Sex'].value_counts())

# c) Null Values Bharna
# Age -> Median se, Embarked -> Mode se, Cabin -> Drop (bahut zyada null hai)
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop(columns=['Cabin'], inplace=True)
print("\nNull bharne ke baad:")
print(df.isnull().sum())

# d) Date / Datatype Fix - Yaha Fare ko float confirm karte hai
df['Fare'] = df['Fare'].astype(float)

# e) Outlier Handle - IQR Method se Fare ke outlier
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR
outliers_before = ((df['Fare'] < lower) | (df['Fare'] > upper)).sum()
df = df[(df['Fare'] >= lower) & (df['Fare'] <= upper)]
print(f"\nFare ke Outliers hataye: {outliers_before}")

# 4. BEFORE vs AFTER Table
print("\n--- BEFORE vs AFTER ---")
print(f"Before Shape: (891, 12) | After Shape: {df.shape}")
print(f"Before Nulls: Age me 177 null the | After Nulls: {df.isnull().sum().sum()}")

# 5. Clean CSV Save
df.to_csv('titanic_cleaned.csv', index=False)
print("\nClean file save ho gayi: titanic_cleaned.csv")
df.head()


--- BEFORE CLEANING ---
Shape: (891, 12)

Null Values:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicates: 0

Data Types:
 PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Sex column ke alag-alag values:
Sex
male      577
female    314
Name: count, dtype: int64

--- Messy Sex Values ---
Sex
male      575
female    312
Male        1
M           1
Female      1
F           1
Name: count, dtype: int64

Duplicates hataye: 0

Cleaned Sex Values:
Sex
Male      577
Female    314
Name: count, dtype: int64

Null bharne ke baad:
PassengerId    0
Survived     

/tmp/ipykernel_368/874950361.py:40: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_368/874950361.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try 

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.2500,S
2,3,1,3,"Heikkinen, Miss. Laina",Male,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",Female,35.0,0,0,373450,8.0500,S
5,6,0,3,"Moran, Mr. James",Female,28.0,0,0,330877,8.4583,Q
